In [ ]:
import pandas as pd
from IPython.display import display
from sklearn.model_selection import GroupKFold
from threadpoolctl import threadpool_limits

from study.data import cohort, features, prediction_metadata
from study.io import WORK, load, reset_directory, save, write_table
from study.models import fit_model, predict
from study.statistics import paired_difference, prediction_rows, risk_matrix

In [ ]:
reset_directory(WORK / 'predictions/local')
metadata, x = cohort('eicu').sort_index(), features('eicu')
shared = load(WORK / 'feature_sets.pkl')['shared']
source_fits = load(WORK / 'models/source.pkl')
parameters = source_fits['parameters']

In [ ]:
ids = metadata.index
folds = {}

# Split hospitals into fitting and evaluation folds
for repeat in source_fits['allocation_repeats']:
    outer = GroupKFold(5, shuffle = True, random_state = repeat)
    folds[repeat], rows = ([], [])

    for fold, (train, test) in enumerate(outer.split(ids, groups = metadata.hospital_id)):
        fit_ids, test_ids = (ids[train], ids[test])
        models = {}

        # Fit each tier and score patients from held-out hospitals
        for tier in ('R1', 'R2', 'R3'):
            with threadpool_limits(limits = 1):
                models[tier] = fit_model(
                    x.loc[fit_ids, shared[tier]],
                    metadata.loc[fit_ids, 'in_hospital_mortality'],
                    'xgboost',
                    parameters,
                )
                risk = predict(models[tier], x.loc[test_ids])

            rows.append(
                prediction_rows(
                    prediction_metadata(metadata.loc[test_ids]),
                    risk,
                    domain = 'eicu',
                    tier = tier,
                    repeat = repeat,
                    fold = fold,
                    restriction = 'full',
                    training = 'full',
                )
            )

        folds[repeat].append({'fit': fit_ids, 'test': test_ids, 'models': models})

    write_table(WORK / 'predictions/local' / f'{repeat}.parquet', pd.concat(rows, ignore_index = True))
    
    print('Local hospital allocation', repeat)

save(WORK / 'models/local.pkl', folds)
local = pd.read_parquet(WORK / 'predictions/local')

In [ ]:
# Compare local process increments in all eICU patients and the external test population
rows = []
for population, ids in [
    ('all_eicu', metadata.index),
    ('external_test', metadata.index[metadata.partition.eq('test')])
]:
    # Pair R3 and R2 risks on the same patients
    r3 = risk_matrix(local.loc[local.tier.eq('R3')], ids)
    r2 = risk_matrix(local.loc[local.tier.eq('R2')], ids)

    for equal_hospital in (False, True):
        rows.append(
            dict(
                population = population,
                **paired_difference(
                    prediction_metadata(metadata.loc[ids]), r3, r2, cluster = True, equal_hospital = equal_hospital
                ),
            )
        )

write_table(WORK / 'results/local.parquet', pd.DataFrame(rows))

display(pd.DataFrame(rows).round(5))